# Capstone — Ranking pages for a decline review

*This notebook mirrors the deployed research paper at `paper/index.html`. It reads only the committed receipts in `work/outputs/` and the committed figures in `work/figures/`, so it runs offline — no warehouse access, no token, no 15-minute data build.*

## Abstract

Which pages in a large content portfolio are worth a reviewer's attention first, and can that ordering be justified without overclaiming? This study ranks 67,279 client–content pairs from the FlyRank ML Internship warehouse using a January–February 2026 feature window and a March 2026 label window, where 7,445 pairs are material declines (base rate 0.111). The method fits one degree-2 logistic model on five features and evaluates it with a six-fold, client-grouped out-of-fold split, comparing it against a raw-volume sort and a frozen rule on the same rows. The model's out-of-fold precision at 50 is 0.300, about 2.7x the random floor of 0.111, but a raw sort by prior volume reaches 0.360 at the same depth, so the model is presented as a ranking aid rather than a predictive win. The output is a review queue that orders pages for human inspection, with limits stated up front: one input carries a measurable signal, the top of the queue is the largest pages only, and one forward window cannot establish stability.

In [1]:
# Setup: locate the repo and the committed artifacts. Nothing here reads the warehouse.
import json, os
from pathlib import Path

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/"work"/"notebooks").is_dir() and (p/".git").exists()), None)
if REPO is None:
    raise SystemExit("run this notebook from inside the repository")
os.chdir(REPO)
OUT = REPO/"work"/"outputs"
FIG = REPO/"work"/"figures"

M07 = json.load(open(OUT/"baseline_metrics.json", encoding="utf-8"))
M08 = json.load(open(OUT/"ml08_metrics.json", encoding="utf-8"))
M09 = json.load(open(OUT/"w06_validation_metrics.json", encoding="utf-8"))
M10 = json.load(open(OUT/"w07_action_playbook_metrics.json", encoding="utf-8"))
M11 = json.load(open(OUT/"w08_volume_baseline.json", encoding="utf-8"))
print("repo root:", REPO.name, "(run from inside the repository)")
print("receipts loaded:", ", ".join(sorted(p.name for p in OUT.glob("*.json"))))

repo root: flyrank-assignments (run from inside the repository)
receipts loaded: baseline_metrics.json, ml08_metrics.json, w06_validation_metrics.json, w07_action_playbook_metrics.json, w08_volume_baseline.json


## 1. Introduction

The decision this supports is narrow and concrete: a content team has more pages than review time, and it needs an order in which to open them. The queue replaces one question — which of 67,279 pages to look at first — and claims nothing beyond that.

The work is observational. It measures which pages look like declines under a defined label, using only data available before the decision point. It does not test whether editing a page recovers traffic, because no intervention was run.

In [2]:
print("Unit of analysis:", M09["unit_of_analysis"] if "unit_of_analysis" in M09 else "one (client, content) pair at the decision point")
print("Decision point :", M09["decision_point"])
print("Pool           : %s pairs | %d clients | %s positives | base rate %.4f"
      % ("{:,}".format(M09["pool_rows"]), M09["pool_clients"],
         "{:,}".format(M09["positives"]), M09["base_rate"]))
print("Random floor at depth 50: %.3f (the number any queue has to beat)" % M09["ml07_reference"]["random_floor_p_at_50"])

Unit of analysis: one (client, content) pair at the decision point
Decision point : 2026-03-01
Pool           : 67,279 pairs | 24 clients | 7,445 positives | base rate 0.1107
Random floor at depth 50: 0.111 (the number any queue has to beat)


## 2. Data

Source is the full pseudonymized FlyRank ML Internship warehouse on Hugging Face (gated; accessed with a read token). It is a daily search-performance panel of roughly 79 million rows; the analysis used only the January–March 2026 partitions for a 24-client lane, condensed to one row per (client, content) pair.

Tables used: `fact_content_daily_performance` (daily grain, the feature and label source), `dim_clients` (per-client history start and access flags), `dim_content` (content metadata, joined for age).

Windows: features 2026-01-01 to 2026-02-28 (59 days); decision point 2026-03-01; label 2026-03-01 to 2026-03-31 (31 days). The feature and label windows do not overlap. April and later were never read.

What was excluded, and why: pages under 200 prior impressions (the pool floor); GA4 engagement columns, because only 8 of 104 clients had GA4 live over the feature window; `content_updated_date`, because 78% of the rows carry an update dated after the decision point, so any "days since update" feature would read the future; and the 90-day query table, whose window falls entirely after the decision point. The frame as built holds 119,115 rows across 28 clients; the modelled pool holds 67,279 rows across 24 clients. A March inner join drops 235 rows (0.35%) whose content disappears by the label month — an outcome-conditioned membership effect, measured as minor and described in full in the Limitations section.

In [3]:
f = M09
print("Tables      : fact_content_daily_performance, dim_clients, dim_content")
print("Feature win : %s -> %s | decision %s | label %s -> %s"
      % ("2026-01-01", "2026-02-28", f["decision_point"], "2026-03-01", "2026-03-31"))
print("Frame built : %s rows / %d clients" % ("{:,}".format(f["frame_rows"]), f["frame_clients"]))
print("Pool        : %s rows / %d clients / %s positives / base %.4f"
      % ("{:,}".format(f["pool_rows"]), f["pool_clients"], "{:,}".format(f["positives"]), f["base_rate"]))
ps = f["population_selection"]
print("March inner-join removed %d rows (%.2f%% of the decided pool) -- outcome-conditioned membership"
      % (ps["removed_by_march_inner_join"], 100*ps["removed_share_of_decided_pool"]))
print("Excluded: GA4 columns (8 of 104 clients live) | content_updated_date (78% post-decision) "
      "| fact_content_query_90d (window after decision point) | pages under 200 prior impressions")

Tables      : fact_content_daily_performance, dim_clients, dim_content
Feature win : 2026-01-01 -> 2026-02-28 | decision 2026-03-01 | label 2026-03-01 -> 2026-03-31
Frame built : 119,115 rows / 28 clients
Pool        : 67,279 rows / 24 clients / 7,445 positives / base 0.1107
March inner-join removed 235 rows (0.35% of the decided pool) -- outcome-conditioned membership
Excluded: GA4 columns (8 of 104 clients live) | content_updated_date (78% post-decision) | fact_content_query_90d (window after decision point) | pages under 200 prior impressions


## 3. Methodology

The label is `is_material_decline`: a pair is a decline when it is in the pool (200+ prior impressions), its January+February impressions-per-30-days is at least 500, and its March rate drops 20% or more below that prior rate.

The model is a single degree-2 logistic regression — the five contracted features plus every squared term and every pairwise product — fit once on the whole pool for deployment. The five features are prior volume (`log_impr_prior`), prior click-through rate (`ctr_prior_pct`), prior weighted average position (`wtd_pos_prior`), prior active days (`active_days_prior`), and content age at the decision point (`content_age_days`). The January-to-February slope (`trend_pct_win`) is read to describe a row but never scored, because adding it lowered the honest out-of-fold result. The L2 penalty is 1.0, chosen by an inner client-grouped split inside the training folds only, never on the rows it is reported against.

Validation is a six-fold, client-grouped out-of-fold split: every pool row is held out exactly once, and no client appears on both sides of a fold. The base rate is printed beside every precision figure. Two baselines are scored on the same rows: a raw sort by prior volume, which requires no fitting, and a frozen rule from the earlier baseline work, presented separately because its 0.720 sits on a label whose denominator it shares.

Leakage checks: the feature and label windows are disjoint by construction; permutation importance shuffles each feature inside held-out client blocks; and a deliberate leak of the label's own March volume drove precision at 50 to 1.000, confirming the evaluation can detect a leak when one exists.

In [4]:
print("Label: in_pool (200+ prior impressions) & prior_rate_30d >= 500 & drop_pct <= -20")
print("Features (5): %s" % ", ".join(M09["features_contract"]))
print("Probe (read, not scored): %s -> %.4f P@50 drop when shuffled"
      % ("trend_pct_win", M09["permutation_importance_p50_drop"]["B probe (+trend_pct_win)"]["trend_pct_win"]))
print("Design : degree-2 logistic (squares + all pairwise terms), one deployment fit on all %s rows"
      % "{:,}".format(M09["pool_rows"]))
print("Penalty: L2 = %.1f (inner grouped CV inside training folds) | rejected: prior penalty %.4f (tuned on its own holdout)"
      % (M09["before_after_p50"]["A contract (5 feats)"]["l2_nested"], M08["best_l2"]))
print("Validation: %d-fold client-grouped out-of-fold, every row held out once, base rate beside every P@K"
      % M09["oof_folds"])
lt = M09["leak_trap"]
print("Leak trap: honest P@50 %.3f -> %.3f with the label's March volume added (%.2fx)"
      % (lt["honest_p50"], lt["leaked_p50"], lt["lift"]))

Label: in_pool (200+ prior impressions) & prior_rate_30d >= 500 & drop_pct <= -20
Features (5): log_impr_prior, ctr_prior_pct, wtd_pos_prior, active_days_prior, content_age_days
Probe (read, not scored): trend_pct_win -> 0.0007 P@50 drop when shuffled
Design : degree-2 logistic (squares + all pairwise terms), one deployment fit on all 67,279 rows
Penalty: L2 = 1.0 (inner grouped CV inside training folds) | rejected: prior penalty 0.0001 (tuned on its own holdout)
Validation: 6-fold client-grouped out-of-fold, every row held out once, base rate beside every P@K
Leak trap: honest P@50 0.300 -> 1.000 with the label's March volume added (3.33x)


## 4. Results

On the pooled out-of-fold rows, with the base rate visible, the model and a raw volume sort are close, and the volume sort is ahead at depth 50.

The honest number the queue inherits is 0.300 at precision 50 — roughly three of the first fifty pages are material declines under this label, about 2.7x the random floor of 0.111. A raw sort by prior volume reaches 0.360 at the same depth with no fitting. Per fold, the model averages slightly higher than the volume sort (0.400 against 0.380) inside the same spread of about 0.10, which is why no single depth is treated as a win.

The frozen rule scored 0.720 on this label, but that number shares the January+February denominator with the label and is not a like-for-like comparison; its forward probe, against a label that cannot see January, was 0.340. Both are reported so neither is mistaken for the model's score.

In [5]:
import numpy as np
A = M09["before_after_p50"]["A contract (5 feats)"]
vol = M11["volume_sort"]
base = M09["base_rate"]

rows = [
    ("Random (base rate)", base, base, base),
    ("Raw volume sort (no fitting)", vol["pooled"]["P@20"], vol["pooled"]["P@50"], vol["pooled"]["P@200"]),
    ("Contract model, 5 features (OOF)", A["after_out_of_fold_p20"], A["after_out_of_fold"], A["after_out_of_fold_p200"]),
]
print("POOLED out-of-fold rows | %s rows | base %.4f" % ("{:,}".format(M09["pool_rows"]), base))
print("  %-34s %7s %7s %8s" % ("ranker", "P@20", "P@50", "P@200"))
for name, a, b, c in rows:
    print("  %-34s %7.3f %7.3f %8.3f" % (name, a, b, c))
print()
print("PER-FOLD P@50 (same client-grouped folds as the model)")
print("  %-5s %9s %8s %8s %8s" % ("fold", "rows", "base", "model", "volume"))
for mf, pf in zip(A["per_fold_p50"], vol["per_fold"]):
    print("  %-5d %9s %8.4f %8.3f %8.3f" % (pf["fold"], "{:,}".format(pf["n"]), pf["base"], mf, pf["P@50"]))
print("  %-5s %9s %8s %8.3f %8.3f   (sd model %.3f | volume %.3f)"
      % ("mean", "", "", float(np.mean(A["per_fold_p50"])), vol["per_fold_P@50_mean"],
         M11["model_ml09_out_of_fold"]["per_fold_P@50_sd"], vol["per_fold_P@50_sd"]))
print()
print("Context, different label basis (not like-for-like):")
print("  frozen rule P@50 %.3f on the blended label | forward probe %.3f at base %.3f"
      % (M07["p_at_50"], M07["forward_label_p_at_50"], M07["forward_label_base_rate"]))

POOLED out-of-fold rows | 67,279 rows | base 0.1107
  ranker                                P@20    P@50    P@200
  Random (base rate)                   0.111   0.111    0.111
  Raw volume sort (no fitting)         0.350   0.360    0.405
  Contract model, 5 features (OOF)     0.350   0.300    0.385

PER-FOLD P@50 (same client-grouped folds as the model)
  fold       rows     base    model   volume
  1           603   0.0697    0.260    0.260
  2        10,635   0.0703    0.380    0.300
  3         9,121   0.1487    0.540    0.560
  4        17,915   0.1160    0.360    0.420
  5         2,549   0.0902    0.480    0.460
  6        26,456   0.1131    0.380    0.280
  mean                        0.400    0.380   (sd model 0.098 | volume 0.119)

Context, different label basis (not like-for-like):
  frozen rule P@50 0.720 on the blended label | forward probe 0.340 at base 0.207


### Chart

Model and baselines across pooled precision depths, with the base rate as the floor. The raw volume sort matches the model at depth 20 and exceeds it at 50 and 200.

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ks = [20, 50, 200]
model = [A["after_out_of_fold_p20"], A["after_out_of_fold"], A["after_out_of_fold_p200"]]
volume = [vol["pooled"]["P@20"], vol["pooled"]["P@50"], vol["pooled"]["P@200"]]
fig, ax = plt.subplots(figsize=(7.2, 4.2))
x = np.arange(len(ks)); w = 0.36
ax.bar(x - w/2, volume, w, label="raw volume sort", color="#2b6cb0")
ax.bar(x + w/2, model, w, label="contract model (OOF)", color="#dd6b20")
ax.axhline(base, color="#c53030", lw=1.3, ls="--", label="base rate %.4f" % base)
ax.set_xticks(x); ax.set_xticklabels(["P@%d" % k for k in ks])
ax.set_ylabel("precision"); ax.set_ylim(0, max(max(volume), max(model)) + 0.08)
ax.set_title("Pooled precision by depth: model vs raw-volume sort (n=67,279, base %.4f)" % base, fontsize=10)
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
CMP = FIG/"w08_model_vs_volume.png"
fig.savefig(CMP, dpi=150)
plt.close(fig)
print("figure written:", CMP.relative_to(REPO).as_posix(), "(%.1f KB)" % (CMP.stat().st_size/1e3))
print("the committed queue figure:", (FIG/"w07_decline_rate_by_rank.png").relative_to(REPO).as_posix())

figure written: work/figures/w08_model_vs_volume.png (33.7 KB)
the committed queue figure: work/figures/w07_decline_rate_by_rank.png


## 5. Limitations

One input carries a detectable signal. Under permutation importance inside held-out client blocks, prior volume costs 0.234 of precision at 50 when shuffled; the other four features land between -0.018 and +0.035, which is noise.

The top of the queue is the largest pages only. The middle volume band (500–2,000 impressions per 30 days) holds 52.3% of all declines and receives zero slots in the top 50; its first row appears at rank 9,741. The score correlates 0.80 with raw volume. The queue is honest about the biggest pages and blind to the middle of the pool.

The label contains its own floor. Material decline requires at least 500 prior impressions, so 40.2% of the pool is zero by construction, and the model's only validated input — prior volume — is written on the same quantity as that floor.

One forward window only. Features end 2026-02-28 and the label is March; April was never read, so there is no second window to test stability.

Outcome-conditioned membership. A March inner join removes 235 rows (0.35%) whose content is gone by the label month; the frame's base rate moves by less than 0.0006 across three join readings, so the effect is minor but real.

Excluded fields. `content_updated_date` is excluded because 78% of rows carry a post-decision-point update; GA4 columns are excluded because only 8 of 104 clients had GA4 live over the window. No causal claim is made anywhere: the score says a page looks like a decline, not why, and no intervention was run.

In [7]:
imp = M09["permutation_importance_p50_drop"]["A contract (5 feats)"]
print("Permutation P@50 drop, held-out clients (the only feature that moves the result):")
for k, v in sorted(imp.items(), key=lambda kv: -kv[1]):
    tag = "carries it" if v > 0.05 else "noise"
    print("  %-20s %+0.4f  %s" % (k, v, tag))
lf = M10["label_volume_floor_caveat"]
print()
print("Label floor: prior_rate_30d >= %s. Pool rows under the floor: %s (%.1f%%), all zero by construction."
      % (lf["floor"].split(">= ")[1], "{:,}".format(lf["pool_rows_under_floor"]), 100*lf["share_of_pool_under_floor"]))
qs = M10["queue"]["score_is_near_a_volume_ranking"]
print("Middle band (B) share of all declines: %.1f%% | rows of band B in the top 50: %d | first band-B rank: %s"
      % (100*qs["band_B_share_of_all_positives"], qs["band_B_rows_in_top_cut"], "{:,}".format(qs["band_B_first_rank"])))
print("Spearman(score, prior_rate_30d): %.3f" % qs["spearman_score_vs_prior_rate_30d"])
print("April or later read:", M10["april_or_later_read"])
print("Withheld:", "; ".join(M10["withheld"][:2]))

Permutation P@50 drop, held-out clients (the only feature that moves the result):
  log_impr_prior       +0.2343  carries it
  wtd_pos_prior        +0.0352  noise
  content_age_days     +0.0097  noise
  active_days_prior    +0.0068  noise
  ctr_prior_pct        -0.0180  noise

Label floor: prior_rate_30d >= 500. Pool rows under the floor: 27,068 (40.2%), all zero by construction.
Middle band (B) share of all declines: 52.3% | rows of band B in the top 50: 0 | first band-B rank: 9,741
Spearman(score, prior_rate_30d): 0.802
April or later read: False
Withheld: no second forward window (April never read); no re-weighting to surface the middle volume band: that needs a second window, not a tuned fit


## 6. Ranked recommendations

The playbook turns the ranking into three actions for a human reviewer, following the earlier baseline's vocabulary and adding none.

- `refresh` — top 50 and already sliding. Open these first.
- `protect` — top 50 and holding. Real volume, not currently falling.
- `monitor` — everything else. No action this week.

Actions are assigned only from two facts the evidence supports: which prior-volume band a page sits in, and whether its January-to-February trend is already sliding. Reason codes therefore name only those two facts. The queue never expands or prunes a page, because the earlier work dropped both actions and nothing here tested them.

In [8]:
q = M10["queue"]
print("Queue: %s rows | cut at depth %d (one reviewer's afternoon, at the measured depth)"
      % ("{:,}".format(M10["pool_rows"]), q["cut_depth"]))
print("Action counts:", q["action_counts"])
print("Measured in-sample decline rate by action:", {k: round(v, 4) for k, v in q["action_measured_decline_rate_in_sample"].items()})
print("Reason codes:", ", ".join(q["reason_codes"]))
print("Actions deliberately not used:", ", ".join(q["actions_dropped_and_why"].keys()))
print()
print("No-go list:")
for line in M10["no_go_list"]:
    print("  -", line)

Queue: 67,279 rows | cut at depth 50 (one reviewer's afternoon, at the measured depth)
Action counts: {'monitor': 67229, 'protect': 34, 'refresh': 16}
Measured in-sample decline rate by action: {'monitor': 0.1105, 'protect': 0.1765, 'refresh': 0.75}
Reason codes: A_over_2000_holding, A_over_2000_sliding, B_500_to_2000_holding, B_500_to_2000_sliding, C_under_500_holding, C_under_500_sliding
Actions deliberately not used: expand, prune

No-go list:
  - no automated edit, publish, unpublish, redirect or client notification off this score
  - no causal claim: no intervention was run, so no 'fixing this recovers X'
  - no per-client reporting: 24 clients, one pooled model, no per-client number measured
  - expand and prune stay out until something tests them
  - no forecast past March: one forward window


## 7. Reproducibility

The repository carries the full trail. Each notebook below is the source for one section, and each writes a small machine-readable receipt in `work/outputs/`.

- `work/notebooks/w01_research_question.ipynb` — the question and the lane.
- `work/notebooks/w03_data_contract.ipynb` — the data contract: tables, windows, exclusions.
- `work/notebooks/w04_baseline_score.ipynb` — the frozen rule baseline.
- `work/notebooks/w05_model.ipynb` — the model and the client-grouped holdout.
- `work/notebooks/w06_validation_audit.ipynb` — the honest out-of-fold evaluation and leakage checks.
- `work/notebooks/w07_action_playbook.ipynb` — the ranked queue and the action playbook.

This notebook reads the receipts only, so it reruns offline in seconds. The model's fold seed is 42 with six client-grouped folds; the deployment fit is deterministic (L-BFGS-B from zeros, no sampling). The warehouse is gated on Hugging Face and needs an access request and a read token; no data is included in the repository.

In [9]:
for name in ("baseline_metrics.json", "ml08_metrics.json", "w06_validation_metrics.json",
             "w07_action_playbook_metrics.json", "w08_volume_baseline.json"):
    p = OUT/name
    print("  %-38s %6d bytes" % (name, p.stat().st_size))
print("  seed: %d | folds: %d | deployment fit: deterministic (L-BFGS-B from zeros)"
      % (M09["seed"], M09["oof_folds"]))
print("  warehouse: gated on Hugging Face; access request + read token; no data committed")

  baseline_metrics.json                    2607 bytes
  ml08_metrics.json                        1665 bytes
  w06_validation_metrics.json              5255 bytes
  w07_action_playbook_metrics.json         7155 bytes
  w08_volume_baseline.json                 2638 bytes
  seed: 42 | folds: 6 | deployment fit: deterministic (L-BFGS-B from zeros)
  warehouse: gated on Hugging Face; access request + read token; no data committed


## 8. Acknowledgments and data credit

Built on the FlyRank ML Internship dataset — https://flyrank.ai

All data is pseudonymized. The published site contains only aggregates: no client names, page URLs, query text, or row-level records.

In [10]:
print("Built on the FlyRank ML Internship dataset -> https://flyrank.ai")
print("Aggregates only in the published paper; the ranked queue CSV is regenerated, not published.")

Built on the FlyRank ML Internship dataset -> https://flyrank.ai
Aggregates only in the published paper; the ranked queue CSV is regenerated, not published.
